# OIBSIP Level 2 – Task 3: Fraud Detection

**Objective:** Build a machine-learning model to identify potentially fraudulent transactions.

> **Important:** For the actual internship submission, use the real dataset provided/recommended by OIBSIP. This notebook can use a CSV such as `creditcard.csv`. If no CSV is available, the notebook creates clearly labelled demo data so you can test the workflow; demo results are not real-world results.


## 1. Import libraries

The libraries below are used for data handling, visualisation, and model evaluation.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    precision_recall_fscore_support
)

print("Fraud Detection project started successfully.")
print("Libraries imported successfully.")


## 2. Load the dataset

Upload your dataset as `creditcard.csv` to the notebook's working directory. The target column is commonly named `Class` (1 = fraud, 0 = legitimate). If your file or target column has a different name, adjust the values below.

In [ ]:
DATA_FILE = "creditcard.csv"
TARGET_COLUMN = "Class"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
    print(f"Loaded dataset: {DATA_FILE}")
else:
    # Demo-only data so the notebook can be tested before the real CSV is available.
    # Replace this demo dataset with the real OIBSIP/Kaggle dataset for your submission.
    rng = np.random.default_rng(42)
    n_rows = 2000
    amount = rng.lognormal(mean=3.0, sigma=1.0, size=n_rows)
    time = rng.uniform(0, 172800, size=n_rows)
    feature_a = rng.normal(size=n_rows)
    feature_b = rng.normal(size=n_rows)
    fraud_probability = 1 / (1 + np.exp(-(-5.0 + 0.0007 * amount + 0.5 * feature_a - 0.4 * feature_b)))
    labels = rng.binomial(1, fraud_probability)
    # Ensure both classes exist in the demonstration data.
    labels[0] = 0
    labels[1] = 1
    df = pd.DataFrame({
        "Time": time,
        "Amount": amount,
        "V1": feature_a,
        "V2": feature_b,
        "Class": labels
    })
    print("WARNING: creditcard.csv was not found. Using DEMO data only.")
    print("Upload the real dataset and rerun this cell before submitting.")

display(df.head())
print("Dataset shape:", df.shape)


## 3. Inspect and prepare the data

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nMissing values per column:")
display(df.isnull().sum().to_frame("Missing values"))

print("\\nTarget distribution:")
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        "Set TARGET_COLUMN to the column that marks fraud (1) vs legitimate (0)."
    )

display(df[TARGET_COLUMN].value_counts().rename(index={0: "Legitimate (0)", 1: "Fraud (1)"}).to_frame("Transactions"))


## 4. Explore class balance

Fraud datasets are usually imbalanced, so accuracy alone is not enough. Precision, recall, F1-score, and the confusion matrix are also reported.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=TARGET_COLUMN, data=df)
plt.title("Legitimate vs Fraud Transactions")
plt.xlabel("Target class (0 = legitimate, 1 = fraud)")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.show()

fraud_rate = (df[TARGET_COLUMN] == 1).mean() * 100
print(f"Fraud transactions: {fraud_rate:.3f}% of rows")


## 5. Prepare features and split the data

Rows without a target value are removed. Missing feature values are filled with each column's median. The target column is excluded from the input features.

In [ ]:
df = df.dropna(subset=[TARGET_COLUMN]).copy()

y = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
valid_rows = y.notna()
df = df.loc[valid_rows].copy()
y = y.loc[valid_rows].astype(int)

X = df.drop(columns=[TARGET_COLUMN]).copy()

# Keep numeric columns for this baseline model.
X = X.select_dtypes(include=[np.number])
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

if X.shape[1] == 0:
    raise ValueError("No numeric feature columns found. Check the dataset format.")

if y.nunique() < 2:
    raise ValueError("The target must contain both classes: 0 (legitimate) and 1 (fraud).")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Feature count:", X.shape[1])


## 6. Train a Random Forest model

This is a baseline model. Real projects may need additional validation, threshold tuning, and methods designed for imbalanced data.

In [ ]:
model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print("Model training completed.")


## 7. Evaluate the model

In [ ]:
print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Legitimate (0)", "Fraud (1)"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

if y_test.nunique() == 2:
    print(f"ROC-AUC: {roc_auc_score(y_test, y_score):.4f}")
else:
    print("ROC-AUC unavailable because the test set contains only one class.")


## 8. Feature importance

Feature importance can help show which columns the model relied on most. Importance does not prove that a feature causes fraud.

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(15).to_frame("Importance"))

plt.figure(figsize=(8, 5))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 9. Conclusion

- The notebook loads transaction data and checks the target distribution.
- A Random Forest classifier is trained to distinguish fraud from legitimate transactions.
- Precision, recall, F1-score, confusion matrix, and ROC-AUC (when available) are used to evaluate performance.
- **Before submitting:** rerun the notebook using the real dataset, check that every cell runs, and write your conclusion based on the actual results. Do not report demo-data results as real fraud-detection performance.
